```python
import dash_ag_grid as dag
import pandas as pd
import plotly.express as px
from dash import Dash, Input, Output, callback, dcc, html

#  Load data
df = pd.read_csv("resources/gapminder.csv")
years = sorted(df["year"].unique())  # Get available years

# Simplified Column Definitions
# [
#     {"field": "country"},
#     {"field": "continent"},
#     {"field": "pop"},
#     {"field": "lifeExp"},
#     {"field": "gdpPercap"}
# ]
column_defs = [
    {"field": c}
    for c in ["country", "year", "continent", "pop", "lifeExp", "gdpPercap"]
]

# Simplified years
# [
#     {"label": "1990", "value": 1990},
#     {"label": "2000", "value": 2000},
#     {"label": "2010", "value": 2010}
# ]
years = [{"label": str(y), "value": y} for y in years]


app = Dash(__name__)

app.layout = html.Div(
    [
        html.H1("🌍 Gapminder Explorer", style={"textAlign": "center"}),
        # Selectors Row
        html.Div(
            [
                html.Div(
                    [
                        html.Label("Select Metric: "),
                        dcc.RadioItems(
                            id="metric-radio",
                            options=[
                                {"label": "Population", "value": "pop"},
                                {"label": "Life Expectancy", "value": "lifeExp"},
                                {"label": "GDP per Capita", "value": "gdpPercap"},
                            ],
                            value="lifeExp",
                            inline=True,
                        ),
                    ],
                    style={"flex": "1"},
                ),
                html.Div(
                    [
                        html.Label("Select Year: "),
                        dcc.Dropdown(
                            id="year-dropdown",
                            options=years,
                            value=2007,  # Default year
                            clearable=False,
                        ),
                    ],
                    style={"flex": "1"},
                ),
            ],
            style={"display": "flex", "margin": "20px", "gap": "20px"},
        ),
        # Charts
        html.Div(
            [
                dcc.Graph(id="bar-chart", style={"flex": "1"}),
                dcc.Graph(id="geo-map", style={"flex": "1"}),
            ],
            style={"display": "flex", "gap": "10px"},
        ),
        # Data Table
        html.H3("Raw Data Table"),
        dag.AgGrid(
            id="data-grid",
            columnDefs=column_defs,
            defaultColDef={"sortable": True, "filter": True},
            dashGridOptions={"pagination": True},
        ),
    ],
    style={"padding": "20px", "fontFamily": "sans-serif"},
)


@callback(
    Output("bar-chart", "figure"),
    Output("geo-map", "figure"),
    Output("data-grid", "rowData"),
    Input("metric-radio", "value"),
    Input("year-dropdown", "value"),
)
def update_ui(metric, year):
    # Filter data by the selected year
    dff = df[df["year"] == year]

    # 1. Average Bar Chart
    bar = px.histogram(
        dff,
        x="continent",
        y=metric,
        histfunc="avg",
        color="continent",
        title=f"Avg {metric} by Continent ({year})",
    )

    # 2. Bubble Map
    geo = px.scatter_geo(
        dff,
        locations="country",
        locationmode="country names",
        size=metric,
        color="continent",
        title=f"Global {metric} in {year}",
    )

    # 3. Table Data
    table_data = dff.to_dict("records")

    return bar, geo, table_data


if __name__ == "__main__":
    app.run(debug=True)
```

```python
import dash_ag_grid as dag
import pandas as pd
import plotly.express as px
from dash import Dash, Input, Output, callback, dcc, html

# 1. Data Preparation
df = pd.read_csv("resources/gapminder.csv")
years = sorted(df["year"].unique())

# 2. Simplified Configs
cols = ["country", "year", "continent", "pop", "lifeExp", "gdpPercap"]
column_defs = [{"field": c} for c in cols]

app = Dash(__name__)

# 3. Layout using HTML Components and CSS Styles
app.layout = html.Div(
    style={
        "fontFamily": "Arial, sans-serif",
        "backgroundColor": "#f4f4f9",
        "padding": "20px",
    },
    children=[
        # HTML Header Section
        html.Header(
            style={"textAlign": "center", "marginBottom": "30px"},
            children=[
                html.H1(
                    "Gapminder Data Explorer", style={"color": "#333", "margin": "0"}
                ),
                html.P("Analyze global trends through time", style={"color": "#666"}),
            ],
        ),
        # Controls Section (using CSS Flexbox for layout)
        html.Div(
            className="control-panel",  # You can define '.control-panel' in a separate assets/style.css
            style={
                "display": "flex",
                "gap": "20px",
                "backgroundColor": "white",
                "padding": "20px",
                "borderRadius": "8px",
                "boxShadow": "0 2px 4px rgba(0,0,0,0.1)",
                "marginBottom": "20px",
            },
            children=[
                html.Div(
                    style={"flex": "1"},
                    children=[
                        html.Label(
                            "Metric:",
                            style={
                                "fontWeight": "bold",
                                "display": "block",
                                "marginBottom": "10px",
                            },
                        ),
                        dcc.RadioItems(
                            id="metric-radio",
                            options=[
                                {"label": i, "value": v}
                                for i, v in [
                                    ("Population", "pop"),
                                    ("Life Expectancy", "lifeExp"),
                                    ("GDP/Cap", "gdpPercap"),
                                ]
                            ],
                            value="lifeExp",
                            inline=True,
                            inputStyle={"marginRight": "5px"},
                            labelStyle={"marginRight": "15px"},
                        ),
                    ],
                ),
                html.Div(
                    style={"width": "200px"},
                    children=[
                        html.Label(
                            "Year:",
                            style={
                                "fontWeight": "bold",
                                "display": "block",
                                "marginBottom": "10px",
                            },
                        ),
                        dcc.Dropdown(
                            id="year-dropdown",
                            options=[{"label": str(y), "value": y} for y in years],
                            value=2007,
                            clearable=False,
                        ),
                    ],
                ),
            ],
        ),
        # Main Content (Graphs)
        html.Main(
            style={"display": "flex", "gap": "20px", "marginBottom": "20px"},
            children=[
                dcc.Graph(
                    id="bar-chart",
                    style={
                        "flex": "1",
                        "backgroundColor": "white",
                        "borderRadius": "8px",
                    },
                ),
                dcc.Graph(
                    id="geo-map",
                    style={
                        "flex": "1",
                        "backgroundColor": "white",
                        "borderRadius": "8px",
                    },
                ),
            ],
        ),
        # Data Table Section
        html.Section(
            style={
                "backgroundColor": "white",
                "padding": "20px",
                "borderRadius": "8px",
            },
            children=[
                html.H3("Raw Data View", style={"marginTop": "0"}),
                dag.AgGrid(
                    id="data-grid",
                    columnDefs=column_defs,
                    defaultColDef={"sortable": True, "filter": True, "resizable": True},
                    dashGridOptions={"pagination": True, "paginationPageSize": 10},
                    style={"height": "400px", "width": "100%"},
                ),
            ],
        ),
    ],
)


# 4. Functional Logic (Callbacks)
@callback(
    Output("bar-chart", "figure"),
    Output("geo-map", "figure"),
    Output("data-grid", "rowData"),
    Input("metric-radio", "value"),
    Input("year-dropdown", "value"),
)
def update_app(metric, year):
    dff = df[df["year"] == year]

    # Bar Chart
    fig_bar = px.bar(
        dff.groupby("continent")[metric].mean().reset_index(),
        x="continent",
        y=metric,
        color="continent",
        title=f"Mean {metric} by Continent",
        template="plotly_white",
    )

    # Geo Map
    fig_map = px.scatter_geo(
        dff,
        locations="country",
        locationmode="country names",
        size=metric,
        color="continent",
        title=f"Global {metric} in {year}",
        projection="natural earth",
        template="plotly_white",
    )

    return fig_bar, fig_map, dff.to_dict("records")


if __name__ == "__main__":
    app.run(debug=True)
```

```python
import dash_ag_grid as dag
import dash_bootstrap_components as dbc
import pandas as pd
import plotly.express as px
from dash import Dash, Input, Output, callback, dcc, html

# 1. Load Data
df = pd.read_csv("resources/gapminder.csv")
years_list = sorted(df["year"].unique())

# 2. Setup AG Grid Columns
column_defs = [
    {"field": c}
    for c in ["country", "year", "continent", "pop", "lifeExp", "gdpPercap"]
]

# 3. Initialize App with a Bootstrap Theme
app = Dash(__name__, external_stylesheets=[dbc.themes.FLATLY])

# --- COMPONENT ABSTRACTION (The "Easy" Parts) ---

# 1. Metric Selector Component
metric_selector = dbc.Col(
    [
        html.Label("Select Metric:", className="fw-bold"),
        dbc.RadioItems(
            id="metric-radio",
            options=[
                {"label": "Population", "value": "pop"},
                {"label": "Life Expectancy", "value": "lifeExp"},
                {"label": "GDP per Capita", "value": "gdpPercap"},
            ],
            value="lifeExp",
            inline=True,
        ),
    ],
    width=8,
)

# 2. Year Selector Component
year_selector = dbc.Col(
    [
        html.Label("Select Year:", className="fw-bold"),
        dcc.Dropdown(
            id="year-dropdown",
            options=[{"label": str(y), "value": y} for y in years_list],
            value=2007,
            clearable=False,
        ),
    ],
    width=4,
)

# 3. Combined Control Card
control_card = dbc.Row(
    [
        dbc.Col(
            [
                dbc.Card(
                    dbc.CardBody(dbc.Row([metric_selector, year_selector])),
                    className="shadow-sm mb-4",
                )
            ]
        )
    ]
)

# --- MAIN LAYOUT ---

app.layout = dbc.Container(
    [
        # Header Row
        dbc.Row(
            [
                dbc.Col(
                    html.H1(
                        "🌍 Gapminder Explorer",
                        className="text-center my-4 text-primary",
                    )
                )
            ]
        ),
        # Control Panel (Replaced with the variable)
        control_card,
        # Charts Row
        dbc.Row(
            [
                dbc.Col(dcc.Graph(id="bar-chart"), lg=6, md=12),
                dbc.Col(dcc.Graph(id="geo-map"), lg=6, md=12),
            ],
            className="mb-4",
        ),
        # Table Row
        dbc.Row(
            [
                dbc.Col(
                    [
                        html.H3("📋 Data Breakdown", className="mb-3"),
                        dag.AgGrid(
                            id="data-grid",
                            columnDefs=column_defs,
                            defaultColDef={
                                "sortable": True,
                                "filter": True,
                                "resizable": True,
                            },
                            dashGridOptions={
                                "pagination": True,
                                "paginationPageSize": 10,
                            },
                            style={"height": "400px", "width": "100%"},
                            className="ag-theme-alpine",
                        ),
                    ]
                )
            ],
            className="pb-5",
        ),
    ],
    fluid=True,
    style={"backgroundColor": "#f8f9fa"},
)


@callback(
    Output("bar-chart", "figure"),
    Output("geo-map", "figure"),
    Output("data-grid", "rowData"),
    Input("metric-radio", "value"),
    Input("year-dropdown", "value"),
)
def update_dashboard(metric, year):
    dff = df[df["year"] == year]

    # Bar Chart
    bar = px.histogram(
        dff,
        x="continent",
        y=metric,
        histfunc="avg",
        color="continent",
        title=f"Average {metric} by Continent ({year})",
        template="plotly_white",
    )

    # Bubble Map
    geo = px.scatter_geo(
        dff,
        locations="country",
        locationmode="country names",
        size=metric,
        color="continent",
        hover_name="country",
        title=f"Global {metric} ({year})",
        projection="natural earth",
        template="plotly_white",
    )

    return bar, geo, dff.to_dict("records")


if __name__ == "__main__":
    app.run(debug=True)
